# 1. Setup Python SDK

In [3]:
# ! pip install betfairlightweight -q
# ! pip install thefuzz -q

In [4]:
import os
import betfairlightweight
from dotenv import load_dotenv

load_dotenv()

# --- Configurazione ---
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")
APP_KEY = os.getenv("APP_KEY")


# Percorsi dei file del certificato
CERT_FILE = 'certs/client-2048.crt'
KEY_FILE = 'certs/client-2048.key'



# Inizializza il client API
trading = betfairlightweight.APIClient(
    username=USERNAME,
    password=PASSWORD,
    app_key=APP_KEY,
    certs='certs',  # Cartella contenente i file client-2048.crt e client-2048.key
    locale='italy'
)

def main():
    try:
        # Inizializza la sessione e fa il login sul server .it
        trading.login()
        print("Login con SDK effettuato con successo!")
        print("Session Token:", trading.session_token)

        # Prova di chiamata API: recupera le corse/eventi per verificare la connessione
        event_types = trading.betting.list_event_types()
        print(f"Connessione attiva! Eventi trovati: {len(event_types)}")

        # Logout ordinato
        trading.logout()

    except betfairlightweight.exceptions.BfetError as e:
        print(f"Errore Betfair SDK: {e}")

if __name__ == '__main__':
    main()

Login con SDK effettuato con successo!
Session Token: OfOJNUdnbTAyaYrtvDz49mrJO2xAIAN0kPzkoQ3Al7M=
Connessione attiva! Eventi trovati: 4


# 2. Metodi Disponibili

In [5]:
from betfairlightweight.filters import (
    market_filter,
    price_projection
)

def check_account_funds():
    """1. VERIFICA SALDO DISPONIBILE"""
    print("\n=== 1. VERIFICA SALDO ===")
    account_funds = trading.account.get_account_funds()
    print(f"Saldo Disponibile: {account_funds.available_to_bet_balance} €")
    print(f"Punti Betfair: {account_funds.discount_rate}")


def search_events():
    """2. RICERCA EVENTI (Es. Calcio)"""
    print("\n=== 2. RICERCA PROSSIME PARTITE DI CALCIO ===")
    
    # EventTypeID '1' corrisponde al Calcio (Soccer)
    soccer_filter = market_filter(event_type_ids=['1'])
    
    events = trading.betting.list_events(filter=soccer_filter)
    
    print(f"Trovati {len(events)} eventi di calcio attivi.")
    for event_object in events[:3]:  # Mostra i primi 3
        event = event_object.event
        print(f" -> ID: {event.id} | Evento: {event.name} | Data: {event.open_date}")
    
    # Restituisce l'ID del primo evento per lo step successivo
    return events[0].event.id if events else None


def get_market_prices(event_id):
    """3. LETTURA DELLE QUOTE (Puntata / Bancata)"""
    print(f"\n=== 3. RECUPERO QUOTE PER EVENTO ID {event_id} ===")
    
    # Cerchiamo il mercato 'MATCH_ODDS' (Esito Finale 1X2)
    m_filter = market_filter(
        event_ids=[event_id],
        market_type_codes=['MATCH_ODDS']
    )
    
    market_catalogues = trading.betting.list_market_catalogue(
        filter=m_filter,
        max_results=1,
        market_projection=['RUNNER_METADATA']
    )
    
    if not market_catalogues:
        print("Nessun mercato MATCH_ODDS trovato per questo evento.")
        return None, None

    market = market_catalogues[0]
    market_id = market.market_id
    print(f"Mercato trovato: {market.market_name} (ID: {market_id})")
    
    # Mappa delle selezioni (es. 1, X, 2)
    runners_dict = {r.selection_id: r.runner_name for r in market.runners}

    # Richiediamo i prezzi correnti (Price Projection)
    price_filter = price_projection(price_data=['EX_BEST_OFFERS'])
    
    market_books = trading.betting.list_market_book(
        market_ids=[market_id],
        price_projection=price_filter
    )
    
    if market_books:
        book = market_books[0]
        print("\n--- QUOTE CORRENTI ---")
        for runner in book.runners:
            name = runners_dict.get(runner.selection_id, 'Sconosciuto')
            
            # Miglior quota PUNTATA (Back)
            best_back = runner.ex.available_to_back[0].price if runner.ex.available_to_back else 'N/A'
            # Miglior quota BANCATA (Lay)
            best_lay = runner.ex.available_to_lay[0].price if runner.ex.available_to_lay else 'N/A'
            
            print(f" {name:<20} | Back (Punta): {best_back} | Lay (Banca): {best_lay}")
            
        # Restituisce market_id e selection_id del primo esito
        first_selection_id = book.runners[0].selection_id
        return market_id, first_selection_id

    return None, None


def place_and_cancel_order(market_id, selection_id):
    """4. PIAZZAMENTO E CANCELLAZIONE SCOMMESSA"""
    print("\n=== 4. ESEMPIO DI PIAZZAMENTO SCOMMESSA ===")
    
    # Definizione dell'ordine: Puntata (BACK) di 2€ a quota irrealistica (1.01)
    # per evitare che venga abbinata accidentalmente durante il test
    limit_order = betfairlightweight.filters.limit_order(
        size=2.00,       # Importo min per Betfair.it
        price=1.01,      # Quota di sicurezza
        persistence_type='LAPSE'
    )
    
    instruction = betfairlightweight.filters.place_instruction(
        order_type='LIMIT',
        selection_id=selection_id,
        side='BACK',      # 'BACK' per Puntare, 'LAY' per Bancare
        limit_order=limit_order
    )
    
    print(f"Invio ordine di prova su Market ID {market_id}...")
    
    place_execution_report = trading.betting.place_orders(
        market_id=market_id,
        instructions=[instruction]
    )
    
    status = place_execution_report.status
    print(f"Stato esecuzione ordine: {status}")
    
    if status == 'SUCCESS':
        instruction_report = place_execution_report.instruction_reports[0]
        bet_id = instruction_report.bet_id
        print(f" Scommessa piazzata con successo! Bet ID: {bet_id}")
        
        # CANCELLAZIONE DELL'ORDINE APPENA PIAZZATO
        print(f"Cancellazione dell'ordine {bet_id} in corso...")
        cancel_instruction = betfairlightweight.filters.cancel_instruction(bet_id=bet_id)
        
        cancel_execution_report = trading.betting.cancel_orders(
            market_id=market_id,
            instructions=[cancel_instruction]
        )
        print(f"Stato cancellazione: {cancel_execution_report.status}")


def main():
    try:
        # Autenticazione
        trading.login()
        print(" Autenticazione eseguita con successo!")

        # 1. Saldo
        check_account_funds()

        # 2. Ricerca Eventi
        event_id = search_events()

        if event_id:
            # 3. Prezzi/Quote
            market_id, selection_id = get_market_prices(event_id)
            
            # 4. Scommessa di Prova (Opzionale - decommenta per testare il piazzamento reale)
            # if market_id and selection_id:
            #     place_and_cancel_order(market_id, selection_id)

    except betfairlightweight.exceptions.BfetError as e:
        print(f"\n Errore durante l'esecuzione: {e}")

    finally:
        # Chiusura sessione
        trading.logout()
        print("\n Sessione chiusa.")


if __name__ == '__main__':
    main()

 Autenticazione eseguita con successo!

=== 1. VERIFICA SALDO ===
Saldo Disponibile: 0.0 €
Punti Betfair: 0.0

=== 2. RICERCA PROSSIME PARTITE DI CALCIO ===
Trovati 402 eventi di calcio attivi.
 -> ID: 35931694 | Evento: Parma v Catania | Data: 2026-08-14 16:00:00+00:00
 -> ID: 35898927 | Evento: Arges Pitesti v Farul Constanta | Data: 2026-08-14 18:30:00+00:00
 -> ID: 35931689 | Evento: Napoli v Aris | Data: 2026-08-12 19:00:00+00:00

=== 3. RECUPERO QUOTE PER EVENTO ID 35931694 ===
Mercato trovato: Match Odds (ID: 1.261043024)

--- QUOTE CORRENTI ---
 Parma                | Back (Punta): 1.3 | Lay (Banca): 1.38
 Catania              | Back (Punta): 1.01 | Lay (Banca): 14.5
 The Draw             | Back (Punta): 4.6 | Lay (Banca): 6.4

 Sessione chiusa.


# 3. Esempio scommessa da stringa del Bet Master

In [6]:
import re
from rapidfuzz import process, fuzz
from datetime import datetime, timedelta
from betfairlightweight.filters import market_filter, time_range

# Messaggio di input (MOCK DEL MESSAGGIO DEL BET MASTER CON UNA PARTITA DI STASERA)
raw_message = """🎯 Bollamaker 🎯

📅 12-08-26 19:30
🆚 FC Copenaghen - Debrecen
🏆 League: Conference League
🎲 Market: 2
💸 Quota: 1.55
🆔 Match ID: 19781446"""


def parse_signal_message(text):
    """
    Estrae dati, orario, squadre, campionato e segno.
    """
    teams_match = re.search(r"🆚\s*(.*?)\s*-\s*(.*)", text)
    league_match = re.search(r"🏆\s*League:\s*(.*)", text)
    market_match = re.search(r"🎲\s*Market:\s*(.*)", text)
    date_match = re.search(r"📅\s*(\d{2}-\d{2}-\d{2}\s+\d{2}:\d{2})", text)

    parsed_date = None
    if date_match:
        parsed_date = datetime.strptime(date_match.group(1), "%d-%m-%y %H:%M")

    return {
        'match_datetime': parsed_date,
        'home_team': teams_match.group(1).strip() if teams_match else None,
        'away_team': teams_match.group(2).strip() if teams_match else None,
        'league': league_match.group(1).strip() if league_match else None,
        'sign': market_match.group(1).strip() if market_match else None
    }


def find_competition_id(trading, league_name, score_threshold=50):
    """
    Cerca la competizione/campionato su Betfair tramite Fuzzy Matching.
    """
    if not league_name:
        return None

    print(f"\n[Step 1] Ricerca del campionato '{league_name}' su Betfair...")
    
    # Recupera tutte le competizioni di calcio attive
    competitions = trading.betting.list_competitions(
        filter=market_filter(event_type_ids=['1'])
    )

    if not competitions:
        print(" Nessuna competizione trovata su Betfair.")
        return None

    # Mappa { "Nome Competizione": ID Competizione }
    comp_map = {c.competition.name: c.competition.id for c in competitions}

    # Fuzzy matching per trovare il campionato più simile
    matches = process.extract(
        league_name,
        comp_map.keys(),
        scorer=fuzz.token_set_ratio,
        limit=3
    )

    if matches:
        best_comp_name, best_score = matches[0][0], matches[0][1]
        print(f"Campionato più simile: '{best_comp_name}' (Somiglianza: {round(best_score, 1)}%)")

        if best_score >= score_threshold:
            comp_id = comp_map[best_comp_name]
            print(f" Campionato identificato! ID: {comp_id}")
            return comp_id

    print(" Campionato non identificato con certezza. Si procederà senza filtro competizione.")
    return None


def find_unique_event_two_step(trading, home_team, away_team, league, match_dt, window_hours=12):
    """
    Trova la partita in 2 step: prima individua il campionato, poi filtra per orario + ID campionato
    e applica il fuzzy matching sulle squadre.
    """
    # 1. STEP 1: Trova l'ID del Campionato
    competition_id = find_competition_id(trading, league)

    # 2. Definizione della finestra temporale per l'orario
    if match_dt:
        from_time = (match_dt - timedelta(hours=window_hours)).strftime("%Y-%m-%dT%H:%M:%SZ")
        to_time = (match_dt + timedelta(hours=window_hours)).strftime("%Y-%m-%dT%H:%M:%SZ")
        time_filter = time_range(from_=from_time, to=to_time)
        print(f"\n[Step 2] Finestra oraria: Da {from_time} a {to_time}")
    else:
        time_filter = None

    # 3. Costruzione del filtro per Betfair con eventuale ID competizione
    comp_ids = [competition_id] if competition_id else None

    m_filter = market_filter(
        event_type_ids=['1'],
        market_start_time=time_filter,
        competition_ids=comp_ids  # <--- Filtro applicato se il campionato è stato trovato
    )

    events = trading.betting.list_events(filter=m_filter)

    # Fallback: Se con il filtro competizione non trova nulla, riprova senza competizione
    if not events and competition_id:
        print(" Nessun evento trovato con il filtro competizione. Riprovo rimuovendo il filtro campionato...")
        m_filter = market_filter(event_type_ids=['1'], market_start_time=time_filter)
        events = trading.betting.list_events(filter=m_filter)

    if not events:
        print(" Nessun evento trovato nel palinsesto Betfair.")
        return None

    print(f"Partite trovate da analizzare: {len(events)}")

    # 4. STEP 2: Fuzzy matching basato solo sul NOME DELLE SQUADRE
    target_teams_query = f"{home_team} {away_team}"
    event_map = {e.event.name: e.event for e in events}

    matches = process.extract(
        target_teams_query,
        event_map.keys(),
        scorer=fuzz.token_set_ratio,
        limit=3
    )

    if not matches:
        return None

    best_match_str, best_score = matches[0][0], matches[0][1]
    print(f"Miglior partita trovata: '{best_match_str}' (Affinità squadre: {round(best_score, 1)}%)")

    if best_score >= 45:
        return event_map[best_match_str]
    
    return None


# --- ESEMPIO DI ESECUZIONE ---
parsed_signal = parse_signal_message(raw_message)

if parsed_signal['match_datetime']:
    print(f"Data e ora estratte: {parsed_signal['match_datetime']}")

# Invocazione del nuovo flusso a 2 passaggi
trading.login()
event = find_unique_event_two_step(
    trading,
    home_team=parsed_signal['home_team'],
    away_team=parsed_signal['away_team'],
    league=parsed_signal['league'],
    match_dt=parsed_signal['match_datetime'],
    window_hours=12 # Perchè la partita è stasera
)

if event:
    print(f"\n RISULTATO FINALE: Partita trovata! Name: '{event.name}' | ID: {event.id}")
else:
    print("\n RISULTATO FINALE: Impossibile trovare la partita.")

trading.logout()

Data e ora estratte: 2026-08-12 19:30:00

[Step 1] Ricerca del campionato 'Conference League' su Betfair...
Campionato più simile: 'UEFA Europa Conference Qualifiers' (Somiglianza: 74.1%)
 Campionato identificato! ID: 12351533

[Step 2] Finestra oraria: Da 2026-08-12T07:30:00Z a 2026-08-13T07:30:00Z
Partite trovate da analizzare: 3
Miglior partita trovata: 'FC Copenhagen v Debreceni VSC' (Affinità squadre: 82.4%)

 RISULTATO FINALE: Partita trovata! Name: 'FC Copenhagen v Debreceni VSC' | ID: 35917314


<LogoutResource>